# pycograd — notebook demo

`%load_ext pycograd` loads [pipescript](https://github.com/smacke/pipescript) (if needed) and enables the DSL. Every model below follows one shape:

```python
with params{ ... } as weights:     # build params, inject them as ambient names
    forward = $ |> ...             # write the forward ONCE, as a pipeline
    objective = lambda: forward(X) |> loss($, target)
    for _ in range(steps):
        value, grads = weights.grad(objective)   # bind weights -> Vars, backprop
        weights.step(grads, lr)                  # in-place SGD
```

No `Var` to construct, no parameters threaded by hand. We open with a linear classifier and an MLP, then build up to **highway networks**, the **Adam** optimizer, **self-attention**, and a full **Transformer encoder block** — all in the same pipeline style, all differentiated through raw numpy.

**Requires** the `pipescript` extra: `pip install pycograd[notebook]`.

In [1]:
%load_ext pycograd

## The primitive: `grad`

Underneath the DSL, `grad` / `value_and_grad` differentiate an ordinary numpy function — the array argument is lifted onto the tape for you.

In [2]:
import numpy as np
from pycograd import grad

def f(x):
    return np.sum(np.sin(x * x))

x = np.array([0.5, 1.0, 1.5])
(gx,) = grad(f)(x)
print("autodiff :", np.round(gx, 4))
print("analytic :", np.round(2 * x * np.cos(x * x), 4))

autodiff : [ 0.9689  1.0806 -1.8845]
analytic : [ 0.9689  1.0806 -1.8845]


## Setup: data and a loss

There is no op library to import for the *model* — pycograd differentiates raw numpy, so activations are just numpy used as pipe stages: `relu` is `np.maximum(0.0, $)`, and a linear layer is `$ @ w + b`.

For the **loss** we reach for the library's first-class `cross_entropy` — mean soft-target softmax cross-entropy *from logits*, a fused, finite-difference-checked op — imported straight from `pycograd`. (Writing it by hand also works: any `np.*` chain differentiates as long as it is a pipe stage or sits inside an instrumented function.)

In [3]:
from pycograd import cross_entropy   # mean soft-target softmax CE, from logits

rng = np.random.default_rng(0)
m = 40
centers = np.array([[2.0, 2.0], [-2.0, 2.0], [0.0, -2.5]])
X = np.vstack([rng.normal(c, 0.5, (m, 2)) for c in centers])
labels = np.repeat(np.arange(3), m)
Y = np.eye(3)[labels]   # one-hot, 3 classes

## Example 1 — a linear softmax classifier

The forward is just `$ @ w + b` (the logits); the loss does the softmax.

In [4]:
with params{
    w = np.zeros((2, 3))
    b = np.zeros(3)
} as weights:
    forward = $ |> $ @ w + b
    objective = |> X |> forward |> cross_entropy($, Y)
    first = last = None
    for _ in range(200):
        value, grads = weights.grad(objective)
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, 0.5)
    logits = forward(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(np.argmax(logits, axis=1) == labels))

loss 1.0986 -> 0.0040
train accuracy: 1.0


## Example 2 — a 2-layer MLP, as one pipeline

`relu` is inlined as the pipe stage `np.maximum(0.0, $)` — no helper needed.

In [5]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> np.maximum(0.0, $) |> $ @ w2 + b2
    objective = |> X |> forward |> cross_entropy($, Y)
    first = last = None
    for _ in range(300):
        value, grads = weights.grad(objective)
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, 0.5)
    logits = forward(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(np.argmax(logits, axis=1) == labels))

loss 1.5142 -> 0.0007
train accuracy: 1.0


## Example 3 — a frozen parameter

`frozen[...]` holds a weight fixed: its gradient comes back `None` and `weights.step` skips it. Here the first-layer bias never moves while everything else trains.

In [6]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = frozen[np.zeros(16)]
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> np.maximum(0.0, $) |> $ @ w2 + b2
    objective = |> X |> forward |> cross_entropy($, Y)
    for _ in range(300):
        value, grads = weights.grad(objective)
        weights.step(grads, 0.5)
    b1_after = weights["b1"].value
    logits = forward(X)

print("final loss        :", round(float(value), 4))
print("b1 stayed all-zero:", bool(np.all(b1_after == 0.0)))
print("train accuracy    :", np.mean(np.argmax(logits, axis=1) == labels))

final loss        : 0.0008
b1 stayed all-zero: True
train accuracy    : 1.0


## Beyond the basics — a small layer toolkit

The examples so far inlined every layer. The fancier models below reuse a few **first-class ops imported straight from `pycograd`** — `relu` / `sigmoid` activations, `layer_norm`, and scaled dot-product `attention` (the same composed, finite-difference-checked functions the library ships, differentiated transparently whether a `Var` or a plain array flows through).

The training loop itself is `train(...)` from `pycograd.examples` — the very same loop as the basic examples above, packaged so it also drives a *stateful* optimizer like Adam in place.

In [7]:
from pycograd import Adam, cosine_decay, layer_norm, relu, sigmoid
from pycograd import scaled_dot_product_attention as attention
from pycograd.examples import train

## Example 4 — a highway network

A [highway layer](https://arxiv.org/abs/1505.00387) learns a per-unit gate `t` and mixes a nonlinear transform with a straight-through copy of its input:

$$y = t \odot \mathrm{relu}(xW_h + b_h) + (1 - t) \odot x.$$

That shape is a natural fit for pipescript's **`fork[...]`**, which sends the piped value down several branches and collects the results into a tuple. `fork[$, ..., ...]` forks the input into the **carry** (`$`, untouched), the **transform** `H(x)`, and the **gate** `t`; the tuple pipe `*|>` then splats that triple into the next stage, where `($x, $h, $t)` names the branches (positionally, matching the fork order) and the combine refers to them by name. The transform-gate bias starts negative, so each layer begins near the identity (`t ≈ 0`, carry the input) and *opens* only where depth helps. We project the 2-D input to the hidden width, stack two highway layers, then read out three class logits.

In [8]:
with params{
    Wp = 0.5 * rng.standard_normal((2, 16))     # input projection
    bp = np.zeros(16)
    Wh1 = 0.5 * rng.standard_normal((16, 16))   # highway layer 1
    bh1 = np.zeros(16)
    Wt1 = 0.5 * rng.standard_normal((16, 16))
    bt1 = -1.0 * np.ones(16)                    # gate biased toward "carry"
    Wh2 = 0.5 * rng.standard_normal((16, 16))   # highway layer 2
    bh2 = np.zeros(16)
    Wt2 = 0.5 * rng.standard_normal((16, 16))
    bt2 = -1.0 * np.ones(16)
    Wo = 0.5 * rng.standard_normal((16, 3))     # readout
    bo = np.zeros(3)
} as weights:
    # each highway layer: fork into (carry, transform, gate), then combine by name
    highway1 = ($ |> fork[$, $ @ Wh1 + bh1 |> relu, $ @ Wt1 + bt1 |> sigmoid]
                  *|> ($x, $h, $t) *|> $h * $t + $x * (1 - $t))
    highway2 = ($ |> fork[$, $ @ Wh2 + bh2 |> relu, $ @ Wt2 + bt2 |> sigmoid]
                  *|> ($x, $h, $t) *|> $h * $t + $x * (1 - $t))
    forward = ($ |> $ @ Wp + bp |> relu |> highway1 |> highway2 |> $ @ Wo + bo)
    objective = |> X |> forward |> cross_entropy($, Y)
    first, last = train(weights, objective, 300, 0.3)
    logits = forward(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(np.argmax(logits, axis=1) == labels))

loss 1.8448 -> 0.0007
train accuracy: 1.0


## Example 5 — training with Adam

`weights.grad(objective)` only computes gradients, so *any* optimizer can consume them. Here we swap in-place SGD for **Adam**, which keeps running estimates of the gradient's first and second moments. `train(...)` detects the `Adam` instance and applies its update (writing the new values back into the live params). Same highway model, far fewer steps to a lower loss.

In [9]:
with params{
    Wp = 0.5 * rng.standard_normal((2, 16))
    bp = np.zeros(16)
    Wh = 0.5 * rng.standard_normal((16, 16))
    bh = np.zeros(16)
    Wt = 0.5 * rng.standard_normal((16, 16))
    bt = -1.0 * np.ones(16)
    Wo = 0.5 * rng.standard_normal((16, 3))
    bo = np.zeros(3)
} as weights:
    highway = ($ |> fork[$, $ @ Wh + bh |> relu, $ @ Wt + bt |> sigmoid]
                 *|> ($x, $h, $t) *|> $h * $t + $x * (1 - $t))
    forward = ($ |> $ @ Wp + bp |> relu |> highway |> $ @ Wo + bo)
    objective = |> X |> forward |> cross_entropy($, Y)
    first, last = train(weights, objective, 120, Adam(lr=0.05))
    logits = forward(X)

print("loss %.4f -> %.4f  (120 Adam steps)" % (first, last))
print("train accuracy:", np.mean(np.argmax(logits, axis=1) == labels))

loss 1.5051 -> 0.0000  (120 Adam steps)
train accuracy: 1.0


## Example 6 — a self-attention sequence classifier

Now the inputs are short **sequences** (each a `length × d` matrix) and the label depends on the tokens jointly. A self-attention layer lets every position attend to every other — $\mathrm{softmax}(QK^\top/\sqrt{d})\,V$ — before we mean-pool over positions and read out a class. Attention is just the `attention` helper; the `Q/K/V` projections are ordinary weight matrices.

Because each example is its own `length × d` sequence, the objective sums the per-example loss over the batch (a plain Python loop — `Var`s add just like numbers).

In [10]:
# synthetic sequences: two classes separated by a per-token mean shift
def make_sequences(n_per=12, L=4, d=8, seed=7):
    g = np.random.default_rng(seed)
    seqs, labels_ = [], []
    for cls, shift in enumerate((-0.7, 0.7)):
        for _ in range(n_per):
            seqs.append(g.normal(shift, 0.6, size=(L, d)))
            labels_.append(cls)
    return seqs, np.array(labels_)

S, S_labels = make_sequences()
S_oh = np.eye(2)[S_labels]
d_model = S[0].shape[1]

with params{
    Wq = 0.3 * rng.standard_normal((d_model, d_model))
    Wk = 0.3 * rng.standard_normal((d_model, d_model))
    Wv = 0.3 * rng.standard_normal((d_model, d_model))
    Wc = 0.3 * rng.standard_normal((d_model, 2))   # classifier head
    bc = np.zeros(2)
} as weights:
    # project x into q/k/v ($x reuses the piped value), attend, mean-pool, read out
    forward1 = ($ |> attention($x @ Wq, $x @ Wk, $x @ Wv) |> np.mean($, axis=0) |> $ @ Wc + bc)
    def objective():
        total = 0.0
        for i in range(len(S)):
            total = total + (forward1(S[i]) |> cross_entropy($, S_oh[i]))
        return total / len(S)
    first, last = train(weights, objective, 150, Adam(lr=0.05))
    preds = np.array([int(np.argmax(forward1(s))) for s in S])

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(preds == S_labels))

loss 0.7085 -> 0.0000
train accuracy: 1.0


## Example 7 — a Transformer encoder block

The capstone composes everything: self-attention with an output projection, a residual connection and `layer_norm`, then a position-wise feed-forward network with its own residual + norm — one standard Transformer encoder block. We train it on the same sequences with Adam and a **cosine-decayed** learning rate (a schedule is just a `callable(step) -> lr`).

In [11]:
def transformer_block(x, Wq, Wk, Wv, Wo, g1, beta1, W1, c1, W2, c2, g2, beta2):
    a = attention(x @ Wq, x @ Wk, x @ Wv) @ Wo   # self-attention + output proj
    x = layer_norm(x + a, g1, beta1)             # residual + norm
    ff = relu(x @ W1 + c1) @ W2 + c2             # position-wise feed-forward
    return layer_norm(x + ff, g2, beta2)         # residual + norm

d_ff = 16
with params{
    Wq = 0.3 * rng.standard_normal((d_model, d_model))
    Wk = 0.3 * rng.standard_normal((d_model, d_model))
    Wv = 0.3 * rng.standard_normal((d_model, d_model))
    Wo = 0.3 * rng.standard_normal((d_model, d_model))   # attention output proj
    g1 = np.ones(d_model)
    beta1 = np.zeros(d_model)
    W1 = 0.3 * rng.standard_normal((d_model, d_ff))      # FFN up
    c1 = np.zeros(d_ff)
    W2 = 0.3 * rng.standard_normal((d_ff, d_model))      # FFN down
    c2 = np.zeros(d_model)
    g2 = np.ones(d_model)
    beta2 = np.zeros(d_model)
    Wc = 0.3 * rng.standard_normal((d_model, 2))         # classifier head
    bc = np.zeros(2)
} as weights:
    forward1 = ($ |> transformer_block($, Wq, Wk, Wv, Wo, g1, beta1, W1, c1, W2, c2, g2, beta2)
                  |> np.mean($, axis=0) |> $ @ Wc + bc)
    def objective():
        total = 0.0
        for i in range(len(S)):
            total = total + (forward1(S[i]) |> cross_entropy($, S_oh[i]))
        return total / len(S)
    schedule = cosine_decay(0.05, total_steps=200)
    first, last = train(weights, objective, 200, Adam(lr=schedule))
    preds = np.array([int(np.argmax(forward1(s))) for s in S])

print("loss %.4f -> %.4f  (Adam + cosine schedule)" % (first, last))
print("train accuracy:", np.mean(preds == S_labels))

loss 1.6512 -> 0.0000  (Adam + cosine schedule)
train accuracy: 1.0


## More

The bundled demos (logistic regression, MLP, LayerNorm/Dropout, single-head Transformer block) train from scratch and are gradient-checked against finite differences:

```
python -m pycograd.examples
```